# Anime Face GAN — Colab Training
Run all cells in order. Checkpoints save to Google Drive every 10 epochs.
For the full architecture walkthrough see `notebook.ipynb`.

In [1]:
# test
import sys, os, subprocess

if not os.path.exists('/content/anime-face-gan'):
    subprocess.run(['git', 'clone', 'https://github.com/xavier-oc-programming/anime-face-gan',
                    '/content/anime-face-gan'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/anime-face-gan', check=True)

os.chdir('/content/anime-face-gan')

# Only install kagglehub — TensorFlow, numpy, PIL etc are pre-installed on Colab.
# Installing the full requirements.txt downgrades numpy to <2.0 which conflicts
# with Colab's TensorFlow binaries and causes a binary incompatibility crash.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=0.3'], check=True)
print('Ready.')

Ready.


In [2]:
# Download dataset
import shutil, kagglehub
from pathlib import Path
from config import DATA_DIR

existing = list(DATA_DIR.glob('*.jpg')) + list(DATA_DIR.glob('*.png'))
if existing:
    print(f'Dataset already present — {len(existing):,} images')
else:
    src = Path(kagglehub.dataset_download('splcher/animefacedataset'))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for p in src.rglob('*.jpg'):
        shutil.copy(p, DATA_DIR / p.name)
    print(f'Copied {len(list(DATA_DIR.glob("*.jpg"))):,} images → {DATA_DIR}')

Dataset already present — 33,180 images


In [3]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
from pathlib import Path
from config import SAVE_INTERVAL

drive.mount('/content/drive')
model_dir   = Path('/content/drive/MyDrive/anime-face-gan/models')
samples_dir = Path('/content/drive/MyDrive/anime-face-gan/samples')
print(f'Checkpoints → {model_dir}')
print(f'Saved every {SAVE_INTERVAL} epochs — a crash loses at most one interval.')

Mounted at /content/drive
Checkpoints → /content/drive/MyDrive/anime-face-gan/models
Saved every 10 epochs — a crash loses at most one interval.


In [4]:
# Train
from train import train
train(model_dir=model_dir, samples_dir=samples_dir)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Loading dataset...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Loaded 33180 images, shape (33180, 64, 64, 3)
Dataset: 260 batches of 128


Model: "generator"

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 128)    │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_6 (LeakyReLU)       │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 512)      │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 4, 4, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_7 (LeakyReLU)       │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         8,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,768,321 (10.56 MB)

 Trainable params: 2,766,529 (10.55 MB)

 Non-trainable params: 1,792 (7.00 KB)

Epoch 001/100 | G: 2.2475 | D: 1.0495 | 82s
Epoch 002/100 | G: 1.8999 | D: 1.0540 | 136s
Epoch 003/100 | G: 1.7774 | D: 1.0473 | 193s
Epoch 004/100 | G: 1.7201 | D: 1.0228 | 275s
Epoch 005/100 | G: 1.6719 | D: 0.9858 | 333s
Epoch 006/100 | G: 1.7957 | D: 0.9231 | 389s
Epoch 007/100 | G: 1.9056 | D: 0.8863 | 446s
Epoch 008/100 | G: 1.9359 | D: 0.9039 | 504s
Epoch 009/100 | G: 1.9075 | D: 0.8907 | 561s
Epoch 010/100 | G: 1.9574 | D: 0.8779 | 617s
  Checkpoint saved → epoch 10 | /content/drive/MyDrive/anime-face-gan/samples/epoch_0010.png
Epoch 011/100 | G: 1.9427 | D: 0.8911 | 674s
Epoch 012/100 | G: 1.9274 | D: 0.8863 | 731s
Epoch 013/100 | G: 1.9362 | D: 0.8901 | 789s
Epoch 014/100 | G: 1.8810 | D: 0.8935 | 846s
Epoch 015/100 | G: 1.8639 | D: 0.9190 | 904s
Epoch 016/100 | G: 1.8235 | D: 0.9157 | 961s
Epoch 017/100 | G: 1.8042 | D: 0.9265 | 1019s
Epoch 018/100 | G: 1.7704 | D: 0.9387 | 1077s
Epoch 019/100 | G: 1.7736 | D: 0.9245 | 1134s
Epoch 020/100 | G: 1.7894 | D: 0.9333 | 1192s
  Ch

In [5]:
# Download generator.keras and training_log.json from Drive after training
from google.colab import files
files.download(str(model_dir / 'generator.keras'))
files.download(str(model_dir / 'training_log.json'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>